# Supplementary tables

Writes supplementary tables from results cached by other notebooks. No figures.

Cheng, Stok Ranen et al., bioRxiv, doi:10.64898/2026.05.14.725081.

Outputs (via `out_path()` into `git_test/data/supp_tables/` and `git_test/data/ref/`):
- `TableS1.csv`: methylation-sensitive TSS clusters (Fig. 5c); preprint Supplementary Table 3
- `TableChiSq.csv`: chi-square screen, KO vs host; Supplementary Table 1
- `TableMotifs_5A.csv`: motif clusters (Fig. 5a); Supplementary Table 2

Notes:
- Inputs are read from the original analysis directory through the repo's `data` link.
- Supplementary Tables 4 (primers, guides) and 5 (cell types) are not written here.

## Initialization

In [ ]:
source("paths.R")

In [2]:
library(dplyr)
library(tibble)

## Supplementary Table 3: methylation-sensitive TSS clusters (Fig. 5c)

- Input: `data/summary/tss_km_{11,10,01}.txt`, from `plt_exp_atac_tss()` (`scripts/Fig5_func.r`, `Fig5.ipynb`).
- Groups: 105 TSSs with accessibility gain and derepression, 562 with accessibility gain only, 391 with induction only; each k-means clustered (k = 10).
- Output: `data/supp_tables/TableS1.csv` (1,058 rows), despite its name Supplementary Table 3.

In [11]:
tss11 = read.table(in_path("./data/summary/tss_km_11.txt"))
tss10 = read.table(in_path("./data/summary/tss_km_10.txt"))
tss01 = read.table(in_path("./data/summary/tss_km_01.txt"))

In [12]:
tss_list = list(tss11 %>% mutate(ATAC_methylation_sensitivity = TRUE, RNA_methylation_sensitivity = TRUE, cluster = paste0("C", cluster)),
                tss10 %>% mutate(ATAC_methylation_sensitivity = TRUE, RNA_methylation_sensitivity = FALSE, cluster = paste0("C", cluster)),
                tss01 %>% mutate(ATAC_methylation_sensitivity = FALSE, RNA_methylation_sensitivity = TRUE, cluster = paste0("C", cluster)))

tss_all = do.call(rbind, tss_list)
rownames(tss_all) = NULL

In [13]:
write.csv(tss_all, out_path("./data/supp_tables/TableS1.csv"))

## Supplementary Table 1: KO vs host expression screen (ED 5a)

- Input: `data/output/chisq_screen.RDS`, from the Chi-square screen section of `prep/ko_vs_host.ipynb`.
- Output: `data/supp_tables/TableChiSq.csv` (Dnmt-TKO and Dnmt-DKO), with `significant = qval <= 0.01`.
- Caveat: the current screen code keeps no `pval` column and its genotype filter also lists TET_TKO, so a regenerated file would not match and `select(pval)` in this section would fail.

In [32]:
chisq = readRDS(in_path("./data/output/chisq_screen.RDS"))
chisq = chisq %>% select(gene, geno, layer, pval, statistic, qval) %>% rename(genotype = geno) %>% mutate(significant = qval <= 0.01)

In [34]:
write.csv(chisq, out_path("./data/supp_tables/TableChiSq.csv"))

## Supplementary Table 2: motif clusters (Fig. 5a)

- Input: `data/output/pwm_trends_km.RDS` (k = 16), from `plot_clust_pwms()` (`scripts/motif_screen_func.r`, `Fig5.ipynb`).
- Output: `data/supp_tables/TableMotifs_5A.csv`.
- Caveat: `plot_clust_pwms_TSS()` saves to the same file name, so after a full `Fig5.ipynb` run the `git_test/` copy holds the TSS clustering. This section reads the original file, also with `use_regenerated_inputs = TRUE`.

In [7]:
km_5A = readRDS("data/output/pwm_trends_km.RDS")  # not in_path(): see caveat
motif_clustering = enframe(sort(km_5A$cluster), name = "motif", value = "cluster") %>% mutate(cluster = paste0("C", cluster))
write.csv(motif_clustering, out_path("./data/supp_tables/TableMotifs_5A.csv"))